# Author: Humza Inam

# Bronze Layer Pipeline - Fraud Detection
**Purpose**: Ingest raw data from source files into Delta tables with minimal transformation

**Tables Created**:
- bronze_transactions
- bronze_fraud_labels
- bronze_cards
- bronze_users
- bronze_mcc_codes

## Setup and Configuration

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import json

In [0]:
%sql
-- Setup catalog if not exists
CREATE CATALOG IF NOT EXISTS databricks_workspace;

-- Setup schemas for medallion architecture, you can also use the GUI
-- Schema == Database
CREATE SCHEMA IF NOT EXISTS databricks_workspace.bronze;
CREATE SCHEMA IF NOT EXISTS databricks_workspace.silver;
CREATE SCHEMA IF NOT EXISTS databricks_workspace.gold;

In [0]:
DATA_PATH = "/Volumes/databricks_workspace/dbo/raw_data/" 
BRONZE_PATH = "databricks_workspace.bronze"  

## 1. Ingest Transactions Data

In [0]:
# Read transactions_data CSV
df_transactions_bronze = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_PATH}transactions_data.csv") 
)

# Write to Bronze Delta table
(df_transactions_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_PATH}.transactions")
)

print(f"Bronze transactions loaded: {df_transactions_bronze.count()} rows")
display(df_transactions_bronze.limit(5))

Bronze transactions loaded: 13305915 rows


id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00.000Z,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01T00:02:00.000Z,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01T00:02:00.000Z,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01T00:05:00.000Z,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01T00:06:00.000Z,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null


## 2. Ingest Fraud Labels

In [0]:
# define fraud labels schema
schema = StructType([
    StructField(
        "target",
        MapType(StringType(), StringType()),
        True
    )
])

# read fraud labels json
df_fraud_labels_bronze = (
    spark.read
         .option("multiLine", "true")
         .schema(schema)
         .json(f"{DATA_PATH}train_fraud_labels.json")
)

# explode fraud labels into rows
df_fraud_labels_bronze = (
    df_fraud_labels_bronze.select(explode("target").alias("transaction_id", "fraud_label"))
)

# Write to Bronze Delta table
(df_fraud_labels_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_PATH}.fraud_labels")
)

print(f"✓ Bronze fraud labels loaded: {df_fraud_labels_bronze.count()} rows")
display(df_fraud_labels_bronze.limit(5))


✓ Bronze fraud labels loaded: 8914963 rows


transaction_id,fraud_label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


## 3. Ingest Cards Data

In [0]:
# read cards data csv
df_cards_bronze = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_PATH}cards_data.csv") 
)

# Write to Bronze Delta table
(df_cards_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_PATH}.cards")
)

print(f"✓ Bronze cards loaded: {df_cards_bronze.count()} rows")
display(df_cards_bronze.limit(5))

✓ Bronze cards loaded: 6146 rows


id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


## 4. Ingest Users Data

In [0]:
df_users_bronze = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_PATH}users_data.csv") 
)

# Write to Bronze Delta table
(df_users_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_PATH}.users")
)

print(f"✓ Bronze users loaded: {df_users_bronze.count()} rows")
display(df_users_bronze.limit(5))

✓ Bronze users loaded: 2000 rows


id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


## 5. Ingest MCC Codes

In [0]:
json_path = f"{DATA_PATH}mcc_codes.json"

# Read and parse JSON
with open(json_path, 'r') as f:
    mcc_dict = json.load(f)

# Convert to DataFrame
import pandas as pd
pdf_mcc = pd.DataFrame([
    {'mcc_code': int(code), 'mcc_description': desc}
    for code, desc in mcc_dict.items()
])

df_mcc_bronze = spark.createDataFrame(pdf_mcc)

# Write to Bronze Delta table
(df_mcc_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_PATH}.mcc_codes")
)

print(f"✓ Bronze MCC codes loaded: {df_mcc_bronze.count()} rows")
display(df_mcc_bronze.orderBy("mcc_code"))

✓ Bronze MCC codes loaded: 109 rows


mcc_code,mcc_description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


## 6. Validation and Summary

In [0]:
# Validate all bronze tables
bronze_tables = ["transactions", "fraud_labels", "cards", "users", "mcc_codes"]

print("=" * 80)
print("BRONZE LAYER VALIDATION SUMMARY")
print("=" * 80)

for table in bronze_tables:
    count = spark.table(f"{BRONZE_PATH}.{table}").count()
    print(f"✓ {BRONZE_PATH}.{table}: {count:,} rows")

print("=" * 80)
print("Bronze layer pipeline completed successfully!")
print("=" * 80)

BRONZE LAYER VALIDATION SUMMARY
✓ databricks_workspace.bronze.transactions: 13,305,915 rows
✓ databricks_workspace.bronze.fraud_labels: 8,914,963 rows
✓ databricks_workspace.bronze.cards: 6,146 rows
✓ databricks_workspace.bronze.users: 2,000 rows
✓ databricks_workspace.bronze.mcc_codes: 109 rows
Bronze layer pipeline completed successfully!


## Data Quality Checks

In [0]:
# Check for null transaction IDs
null_check = spark.table(f"{BRONZE_PATH}.transactions").filter(col("id").isNull()).count()
print(f"Transactions with null IDs: {null_check}")

# Check for duplicate transaction IDs
duplicate_check = (spark.table(f"{BRONZE_PATH}.transactions")
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
    .count()
)
print(f"Duplicate transaction IDs: {duplicate_check}")

# Check fraud label distribution
fraud_dist = (spark.table(f"{BRONZE_PATH}.fraud_labels")
    .groupBy("fraud_label")
    .count()
    .orderBy("fraud_label")
)
print("\nFraud Label Distribution:")
display(fraud_dist)

Transactions with null IDs: 0
Duplicate transaction IDs: 0

Fraud Label Distribution:


fraud_label,count
No,8901631
Yes,13332
